<a href="https://colab.research.google.com/github/nishachari-here/LIGO/blob/main/MLxLIGO_url2_mine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Mouting GDrive
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/My Drive/MLxLIGO/ligoTraining.csv"

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(path)
df.columns = df.columns.str.replace('trainingset_v1d1_metadata.csv/', '', regex=False)

In [ ]:
# Create the 'output' column and assign numerical labels
labels = df['label'].unique()
label_map = {label: i for i, label in enumerate(labels)}
df['output'] = df['label'].map(label_map)

print("Label to number mapping:")
for label, num in label_map.items():
    print(f"{label}: {num}")
df.head()

Label to number mapping:
Whistle: 0
1080Lines: 1
Blip: 2
Violin_Mode: 3
Scattered_Light: 4
Power_Line: 5
Light_Modulation: 6
Koi_Fish: 7
None_of_the_Above: 8
Scratchy: 9
Tomte: 10
Chirp: 11
Wandering_Line: 12
Extremely_Loud: 13
Repeating_Blips: 14
No_Glitch: 15
Low_Frequency_Lines: 16
Paired_Doves: 17
Low_Frequency_Burst: 18
Helix: 19
Air_Compressor: 20
1400Ripples: 21


,event_time,ifo,peak_time,peak_time_ns,start_time,start_time_ns,duration,search,process_id,event_id,...,param_one_name,param_one_value,gravityspy_id,label,sample_type,url1,url2,url3,url4,output
0,1.134216e+09,L1,1134216192,931639909,1134216192,832031011,0.18750,Omicron,0,21,...,phase,-2.72902,zmIdpucyOG,Whistle,train,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,0
1,1.129360e+09,L1,1129359781,558593034,1129359781,47851085,0.94238,Omicron,0,107,...,phase,1.10682,zWFRqqDxwv,Whistle,test,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,0
2,1.127425e+09,L1,1127425468,976317882,1127425468,960937023,0.04688,Omicron,0,218,...,phase,-0.83099,zKCTakFVcf,Whistle,train,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,0
3,1.132637e+09,L1,1132636755,365233898,1132636754,951172113,0.82422,Omicron,0,88,...,phase,0.76242,z14BdoiFZS,Whistle,validation,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,0
4,1.132036e+09,L1,1132035853,197264909,1132035852,933837890,2.00366,Omicron,0,16,...,phase,-0.31161,yyjqLCtAmO,Whistle,validation,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,https://panoptes-uploads.zooniverse.org/produc...,0


In [ ]:
# Count the occurrences of each label
label_counts = df['label'].value_counts()

# Display the counts
print("Occurrences of each label:")
print(label_counts)

Occurrences of each label:
label
Blip                   1821
Koi_Fish                706
Low_Frequency_Burst     621
Light_Modulation        512
Power_Line              449
Extremely_Loud          447
Low_Frequency_Lines     447
Scattered_Light         443
Violin_Mode             412
Scratchy                337
1080Lines               328
Whistle                 299
Helix                   279
Repeating_Blips         263
No_Glitch               150
Tomte                   103
None_of_the_Above        81
1400Ripples              81
Chirp                    60
Air_Compressor           58
Wandering_Line           42
Paired_Doves             27
Name: count, dtype: int64


In [ ]:
df['url2'] = df['url2'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
df[['url2', 'output']].to_csv('/content/drive/MyDrive/MLxLIGO/labels.csv', index=False)

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image, UnidentifiedImageError
import requests
from io import BytesIO
from torchvision import transforms

class SpectrogramDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # Decode the URL and get the label
        url = self.dataframe.iloc[idx]['url2']
        label = self.dataframe.iloc[idx]['output']


        try:
            # Download the image from the URL
            response = requests.get(url, timeout=10)
            response.raise_for_status()  # Raise an exception for bad status codes
            img = Image.open(BytesIO(response.content)).convert('L') # 'L' for grayscale

            # Apply transformations if any
            if self.transform:
                img = self.transform(img)

            return img, label
        except (requests.exceptions.RequestException, UnidentifiedImageError) as e:
            # Handle potential errors during download or image opening
            print(f"Skipping sample at index {idx} due to error: {e}")
            return self.__getitem__((idx + 1) % len(self.dataframe))  # go on to try the next sample

In [ ]:
from torch.utils.data import DataLoader

# Define transformations. Resize to a consistent size, e.g., 64x64
#Spectrograms are 1-channel grayscale images
data_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    #transforms.Normalize(mean=[0.5], std=[0.5])
])

# Create an instance of the dataset
spectrogram_dataset = SpectrogramDataset(dataframe=df, transform=data_transforms)

# Create the DataLoader
dataloader = DataLoader(spectrogram_dataset, batch_size=32, shuffle=True, num_workers=0)

# Now you can iterate through the dataloader to get batches
for images, labels in dataloader:
    print(f"Batch shape: {images.shape}")
    print(f"Labels shape: {labels.shape}")
    # The shape should be [batch_size, channels, height, width] -> [32, 1, 64, 64]
    break

In [ ]:
import torch
import torch.nn as nn

class GravitySpyCNN1(nn.Module):
    def __init__(self, num_classes):
        super(GravitySpyCNN1, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # 8x8 spatial resolution with 128 channels
        self.sequence_length = 8 * 8
        self.embed_dim = 128
        self.attention = nn.MultiheadAttention(embed_dim=self.embed_dim, num_heads=8, batch_first=True)

        # Classifier
        self.fc_layers = nn.Sequential(
            nn.Linear(self.embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # Pass through CNN layers
        x = self.conv_layers(x)

        # Reshape for Attention: (batch, channels, height, width) -> (batch, sequence_length, embed_dim)
        batch_size, channels, h, w = x.shape
        x = x.view(batch_size, channels, h * w).permute(0, 2, 1)

        # Apply attention
        attn_output, _ = self.attention(x, x, x)

        # Aggregate the output (e.g., by taking the mean across the sequence dimension)
        x = torch.mean(attn_output, dim=1)

        # Pass through classifier
        x = self.fc_layers(x)
        return x

In [ ]:
#training loop
from tqdm import tqdm
def train(model, dataloader, optimizer, loss_func, device):
  model.train()
  running_loss = 0.0
  correct = 0
  total = 0

  for batch in tqdm(dataloader):
    # Filter out None values from the batch
    valid_samples = [(images, labels) for images, labels in zip(batch[0], batch[1]) if images is not None and labels is not None]

    if not valid_samples:
        # Skip the batch if it contains no valid samples
        continue

    # Unpack the valid samples
    images, labels = zip(*valid_samples)
    images = torch.stack(images, 0) # Stack the valid images into a tensor
    labels = torch.tensor(labels) # Convert labels to a tensor

    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = loss_func(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

  epoch_loss = running_loss / total if total > 0 else 0.0
  epoch_acc = correct / total if total > 0 else 0.0
  return epoch_loss, epoch_acc

In [ ]:
from torch.utils.data import random_split

train_ratio = 0.8
test_ratio = 1-train_ratio



dataset_size = len(spectrogram_dataset)
train_size = int(train_ratio * dataset_size)
test_size = dataset_size - train_size



train_dataset, test_dataset = random_split(spectrogram_dataset, [train_size, test_size])


train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False) # Shuffle is usually False for the test set

print(f"Dataset split into {len(train_dataset)} training samples, {len(test_dataset)} testing samples.")

In [ ]:

from pathlib import Path

MODEL_PATH = Path("/content/drive/MyDrive/MLxLIGO/trained_models")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs=15
train_loss, train_acc, test_loss, test_acc =[], [], [], []
loss_func = nn.CrossEntropyLoss()
loaded_model_1=GravitySpyCNN1(num_classes=len(label_map)).to(device)
optimizer = torch.optim.Adam(loaded_model_1.parameters(), lr=0.001)
running_loss=0.0
total=0
correct=0
CHECKPOINT_PATH = MODEL_PATH / "checkpoint.pth"

try:
  for epoch in range(epochs):
    tl,ta=train(loaded_model_1, train_dataloader, optimizer, loss_func, device)
    train_loss.append(tl)
    train_acc.append(ta)
    print(f"Epoch {epoch+1}/{epochs}, train Loss: {tl:.4f},train Accuracy: {ta:.4f}")
except Exception as e:
  print(e)
finally:
  torch.save({
      'epoch': epoch,
      'model_state_dict': loaded_model_1.state_dict(),
      'optimizer_state_dict': optimizer.state_dict(),
      'train_loss': train_loss,
      'train_acc': train_acc,
      'test_loss': test_loss,
      'test_acc': test_acc
      }, CHECKPOINT_PATH)


 48%|████▊     | 191/399 [41:20<45:21, 13.09s/it]

Skipping sample at index 6340 due to error: HTTPSConnectionPool(host='panoptes-uploads.zooniverse.org', port=443): Read timed out. (read timeout=10)


100%|██████████| 399/399 [1:26:08<00:00, 12.95s/it]


Epoch 1/15, train Loss: 1.4547,train Accuracy: 0.5497


100%|██████████| 399/399 [32:06<00:00,  4.83s/it]


...

 91%|█████████ | 364/399 [21:36<01:39,  2.83s/it]

Skipping sample at index 7949 due to error: HTTPSConnectionPool(host='panoptes-uploads.zooniverse.org', port=443): Read timed out. (read timeout=10)


100%|██████████| 399/399 [23:56<00:00,  3.60s/it]


Epoch 9/15, train Loss: 0.2072,train Accuracy: 0.9396


 37%|███▋      | 148/399 [08:03<11:32,  2.76s/it]

In [ ]:
# saving model

MODEL_PATH.mkdir(parents=True, exist_ok=True)


MODEL_NAME = "url2CNN.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME


model = GravitySpyCNN1(num_classes=len(label_map)).to(device)

print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model.state_dict(),
           f=MODEL_SAVE_PATH)